# 03. Integración de Fuentes

Este notebook consolida los datos de ambas fuentes (interna y externa) en un único dataset unificado, manteniendo el **Modelo Canónico de 33 columnas**.

**Objetivo:**
Crear el archivo `unified_dataset.csv` que servirá como base para la detección de duplicados y el análisis final.

In [1]:
import pandas as pd
import os

# Configuración de rutas
PROCESSED_PATH = '../data/processed/'

# Carga de datasets limpios
df_int = pd.read_csv(os.path.join(PROCESSED_PATH, 'internal_clean.csv'))
df_ext = pd.read_csv(os.path.join(PROCESSED_PATH, 'external_clean.csv'))

print(f"Registros internos cargados: {df_int.shape}")
print(f"Registros externos cargados: {df_ext.shape}")

Registros internos cargados: (119390, 33)
Registros externos cargados: (119390, 33)


## 1. Unión Vertical (Concatenación)
Como ambos datasets ya comparten el mismo esquema de 33 columnas gracias al proceso de estandarización previo, la unión es directa.

In [2]:
# Agregamos columna de origen para trazabilidad en Record Linkage
df_int['source'] = 'internal'
df_ext['source'] = 'external'

# Concatenamos asegurando que el índice se reinicie
df_unified = pd.concat([df_int, df_ext], ignore_index=True)

print(f"Total de registros en dataset unificado: {df_unified.shape[0]}")
print(f"Total de columnas (incluyendo 'source'): {df_unified.shape[1]}")

Total de registros en dataset unificado: 238780
Total de columnas (incluyendo 'source'): 34


## 2. Verificación de Integridad
Validamos que no se hayan introducido nulos inesperados durante la unión y que las columnas coincidan con el modelo.

In [3]:
# Verificar que todas las columnas del modelo canónico están presentes (33 + 'source')
expected_cols = 34
if len(df_unified.columns) == expected_cols:
    print(f"[OK] El dataset mantiene las {expected_cols} columnas (Modelo Canónico + 'source').")
else:
    print(f"[ADVERTENCIA] Se detectaron {len(df_unified.columns)} columnas. Revisar estandarización.")

# Verificación rápida de nulos post-unión
nulos_totales = df_unified.isnull().sum().sum()
print(f"Total de nulos en el dataset unificado: {nulos_totales}")
print("(Nota: Es normal encontrar nulos en 'email' según la política de limpieza)")

[OK] El dataset mantiene las 34 columnas (Modelo Canónico + 'source').
Total de nulos en el dataset unificado: 341378
(Nota: Es normal encontrar nulos en 'email' según la política de limpieza)


## 3. Exportación
Guardamos el archivo final en la capa de datos procesados.

In [4]:
output_file = os.path.join(PROCESSED_PATH, 'unified_dataset.csv')
df_unified.to_csv(output_file, index=False)

print(f"[RESULTADO] Dataset unificado guardado exitosamente en: {output_file}")

[RESULTADO] Dataset unificado guardado exitosamente en: ../data/processed/unified_dataset.csv
